In [1]:
import six
import torch

# Create a dummy object to serve as torch._six with the required attribute.
class DummySix:
    pass

torch._six = DummySix()
torch._six.string_classes = six.string_types

import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
#import numpy as np
from PIL import Image

def generate_rotating_mnist_time_series(n_samples=100, sequence_length=10, rotate_range=(0, 360), download=True):
    """
    Generates a time series dataset from MNIST digits by rotating each image.

    Parameters:
      n_samples (int): Number of MNIST images to sample.
      sequence_length (int): Number of rotated images in each time series.
      rotate_range (tuple): (min_angle, max_angle) over which to rotate the digit.
      download (bool): Whether to download MNIST dataset if not present.
    
    Returns:
      time_series_dataset (list): List of tuples (time_series, label) where:
                                  - time_series is a tensor of shape (sequence_length, C, H, W)
                                  - label is the MNIST digit label.
    """
    # Define a basic transform to get the tensor (we need to convert to PIL for rotation)
    transform = transforms.ToTensor()
    dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=download)
    
    time_series_dataset = []
    # Randomly select indices from the dataset
    indices = np.random.choice(len(dataset), size=n_samples, replace=False)
    
    # Create equally spaced rotation angles over the specified range
    angles = np.linspace(rotate_range[0], rotate_range[1], sequence_length)
    
    for idx in indices:
        image, label = dataset[idx]
        # Convert the tensor image to PIL Image for rotation
        pil_image = transforms.ToPILImage()(image)
        time_series = []
        # Rotate the image by each angle and convert back to tensor
        for angle in angles:
            rotated_image = pil_image.rotate(angle)
            rotated_tensor = transforms.ToTensor()(rotated_image)
            time_series.append(rotated_tensor)
        # Stack the sequence (shape: [sequence_length, C, H, W])
        time_series = torch.stack(time_series)
        time_series_dataset.append((time_series, label))
    
    return time_series_dataset

In [2]:
dataset_ts = generate_rotating_mnist_time_series(n_samples=5, sequence_length=10, rotate_range=(0, 360))
    
# Plot one time series sample
sample, label = dataset_ts[0]
print("Label:", label)
fig, axs = plt.subplots(1, sample.shape[0], figsize=(15, 2))
for i in range(sample.shape[0]):
    axs[i].imshow(sample[i].squeeze(), cmap='gray')
    axs[i].axis('off')
plt.suptitle("Rotating MNIST Digit (Label {})".format(label))
plt.show()

TypeError: expected np.ndarray (got numpy.ndarray)

In [ ]:
import torch
print(torch.__version__)
